In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import matplotlib.pyplot as plt

df = pd.read_csv('customer_segmentation.csv')
print("✓ Data loaded successfully!")
print(f"Dataset shape: {df.shape}")

print(f"Number of customers: {len(df)}")
print(f"Number of features: {len(df.columns)}")

df['Age'] = 2024 - df['Year_Birth']
spending_columns = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df['TotalSpent'] = df[spending_columns].sum(axis=1)
purchase_columns = ['NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
df['TotalPurchases'] = df[purchase_columns].sum(axis=1)
df['TotalChildren'] = df['Kidhome'] + df['Teenhome']
campaign_columns = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response']
df['TotalCampaignAccepted'] = df[campaign_columns].sum(axis=1)

print(" New features created successfully!")


clustering_features = ['Age', 'Income', 'TotalSpent', 'TotalPurchases', 'Recency', 'TotalChildren', 'TotalCampaignAccepted', 'NumWebVisitsMonth']
X = df[clustering_features].copy()

X = X.fillna(X.mean())

print(f"✓ Selected {len(clustering_features)} features for clustering")
print("Features:", clustering_features)

print("\n--- STEP 4: Standardizing Data ---")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("✓ Data standardized successfully!")


print("\n--- STEP 5: K-MEANS CLUSTERING ---")

best_k = 4 
kmeans = KMeans(n_clusters=best_k, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

df['KMeans_Cluster'] = kmeans_labels

kmeans_silhouette = silhouette_score(X_scaled, kmeans_labels)
kmeans_davies_bouldin = davies_bouldin_score(X_scaled, kmeans_labels)

print(f"K-Means clustering completed with {best_k} clusters")
print(f"Silhouette Score: {kmeans_silhouette:.3f} (higher is better)")
print(f"Davies-Bouldin Score: {kmeans_davies_bouldin:.3f} (lower is better)")

kmeans_cluster_sizes = df['KMeans_Cluster'].value_counts().sort_index()
print("\nK-Means Cluster Sizes:")
for i in range(best_k):
    percentage = (kmeans_cluster_sizes[i] / len(df)) * 100
    print(f"  Cluster {i}: {kmeans_cluster_sizes[i]} customers ({percentage:.1f}%)")

hierarchical = AgglomerativeClustering(n_clusters=4, linkage='ward')
hierarchical_labels = hierarchical.fit_predict(X_scaled)

df['Hierarchical_Cluster'] = hierarchical_labels

hier_silhouette = silhouette_score(X_scaled, hierarchical_labels)
hier_davies_bouldin = davies_bouldin_score(X_scaled, hierarchical_labels)

print(" Hierarchical clustering completed with 4 clusters")
print(f"Silhouette Score: {hier_silhouette:.3f} (higher is better)")
print(f"Davies-Bouldin Score: {hier_davies_bouldin:.3f} (lower is better)")

hier_cluster_sizes = df['Hierarchical_Cluster'].value_counts().sort_index()
print("\nHierarchical Cluster Sizes:")
for i in range(4):
    percentage = (hier_cluster_sizes[i] / len(df)) * 100
    print(f"  Cluster {i}: {hier_cluster_sizes[i]} customers ({percentage:.1f}%)")
print("Performance Comparison:")
print(f"Method                    Silhouette Score    Davies-Bouldin Score")
print(f"K-Means                   {kmeans_silhouette:.3f}               {kmeans_davies_bouldin:.3f}")
print(f"Hierarchical              {hier_silhouette:.3f}               {hier_davies_bouldin:.3f}")
kmeans_analysis = df.groupby('KMeans_Cluster')[clustering_features].mean()
cluster_names = {
    0: "Budget Conscious",
    1: "Average Customers", 
    2: "High Spenders",
    3: "Family Oriented"
}

print("K-Means Cluster Characteristics:")
for cluster_id in range(best_k):
    cluster_data = kmeans_analysis.loc[cluster_id]
    size = kmeans_cluster_sizes[cluster_id]
    percentage = (size / len(df)) * 100
    
    print(f"\n🔸 Cluster {cluster_id} - {cluster_names.get(cluster_id, 'Unknown')} ({size} customers, {percentage:.1f}%)")
    print(f"   Average Age: {cluster_data['Age']:.0f} years")
    print(f"   Average Income: ${cluster_data['Income']:,.0f}")
    print(f"   Average Spending: ${cluster_data['TotalSpent']:,.0f}")
    print(f"   Average Purchases: {cluster_data['TotalPurchases']:.1f}")
    print(f"   Average Children: {cluster_data['TotalChildren']:.1f}")
    print(f"   Campaign Response: {cluster_data['TotalCampaignAccepted']:.2f}")

hier_analysis = df.groupby('Hierarchical_Cluster')[clustering_features].mean()

hier_cluster_names = {
    0: "Mainstream",
    1: "Premium",
    2: "Young Families", 
    3: "Senior Segment"
}

print("Hierarchical Cluster Characteristics:")
for cluster_id in range(4):
    cluster_data = hier_analysis.loc[cluster_id]
    size = hier_cluster_sizes[cluster_id]
    percentage = (size / len(df)) * 100
    
    print(f"\n Cluster {cluster_id} - {hier_cluster_names.get(cluster_id, 'Unknown')} ({size} customers, {percentage:.1f}%)")
    print(f"   Average Age: {cluster_data['Age']:.0f} years")
    print(f"   Average Income: ${cluster_data['Income']:,.0f}")
    print(f"   Average Spending: ${cluster_data['TotalSpent']:,.0f}")
    print(f"   Average Purchases: {cluster_data['TotalPurchases']:.1f}")
    print(f"   Average Children: {cluster_data['TotalChildren']:.1f}")
    print(f"   Campaign Response: {cluster_data['TotalCampaignAccepted']:.2f}")

summary_results = pd.DataFrame({
    'Customer_ID': df['ID'],
    'Age': df['Age'],
    'Income': df['Income'],
    'TotalSpent': df['TotalSpent'],
    'KMeans_Cluster': df['KMeans_Cluster'],
    'Hierarchical_Cluster': df['Hierarchical_Cluster']
})


if kmeans_silhouette > hier_silhouette:
    print(" K-Means clustering performed better based on silhouette score")
    better_method = "K-Means"
else:
    print(" Hierarchical clustering performed better based on silhouette score")
    better_method = "Hierarchical"



✓ Data loaded successfully!
Dataset shape: (2240, 29)
Number of customers: 2240
Number of features: 29
 New features created successfully!
✓ Selected 8 features for clustering
Features: ['Age', 'Income', 'TotalSpent', 'TotalPurchases', 'Recency', 'TotalChildren', 'TotalCampaignAccepted', 'NumWebVisitsMonth']

--- STEP 4: Standardizing Data ---
✓ Data standardized successfully!

--- STEP 5: K-MEANS CLUSTERING ---
K-Means clustering completed with 4 clusters
Silhouette Score: 0.201 (higher is better)
Davies-Bouldin Score: 1.684 (lower is better)

K-Means Cluster Sizes:
  Cluster 0: 1018 customers (45.4%)
  Cluster 1: 418 customers (18.7%)
  Cluster 2: 173 customers (7.7%)
  Cluster 3: 631 customers (28.2%)
 Hierarchical clustering completed with 4 clusters
Silhouette Score: 0.147 (higher is better)
Davies-Bouldin Score: 1.748 (lower is better)

Hierarchical Cluster Sizes:
  Cluster 0: 554 customers (24.7%)
  Cluster 1: 640 customers (28.6%)
  Cluster 2: 678 customers (30.3%)
  Cluster 3: